# FraudShield — Exploratory Data Analysis

Understanding the Credit Card Fraud Detection dataset before building models.

**Dataset**: Kaggle Credit Card Fraud Detection (mlg-ulb/creditcardfraud)

**Goal**: Explore class imbalance, feature distributions, and correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('darkgrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 1. Class Distribution

Highly imbalanced — only 0.17% fraudulent transactions. This will be the main challenge.

In [ ]:
counts = df['Class'].value_counts()
print(f'Legitimate: {counts[0]} ({counts[0]/len(df)*100:.2f}%)')
print(f'Fraudulent: {counts[1]} ({counts[1]/len(df)*100:.4f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
counts.plot.bar(ax=ax1, color=['green', 'red'])
ax1.set_title('Transaction Class Counts')
ax1.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
ax1.set_ylabel('Count')

counts.plot.pie(ax=ax2, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%', colors=['green', 'red'])
ax2.set_title('Class Distribution')
ax2.set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Time Analysis

Time is in seconds from the first transaction. Let's see if fraud clusters at certain times.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(df['Time'], bins=48, edgecolor='k', alpha=0.7)
ax1.set_title('All Transactions Over Time (48 bins)')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Count')

ax2.hist(df[df['Class'] == 1]['Time'], bins=48, color='red', alpha=0.7, edgecolor='k')
ax2.set_title('Fraudulent Transactions Over Time')
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Amount Distribution

Fraud transactions tend to have smaller amounts (card testing), legitimate ones vary more.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(df['Amount'], bins=50, edgecolor='k', alpha=0.7)
ax1.set_title('All Transaction Amounts')
ax1.set_xlabel('Amount')

ax2.hist(df[df['Class'] == 1]['Amount'], bins=30, color='red', alpha=0.7, edgecolor='k')
ax2.set_title('Fraud Transaction Amounts')
ax2.set_xlabel('Amount')

plt.tight_layout()
plt.show()

print('Fraud amount stats:')
print(df[df['Class'] == 1]['Amount'].describe())

## 4. PCA Components (V1–V28)

The dataset features V1–V28 are PCA-transformed to protect confidentiality. Let's compare their distributions between classes.

In [ ]:
pca_cols = [f'V{i}' for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(pca_cols):
    ax = axes[i]
    ax.hist(df[df['Class'] == 0][col], bins=50, alpha=0.5, label='Legit', density=True)
    ax.hist(df[df['Class'] == 1][col], bins=50, alpha=0.5, label='Fraud', color='red', density=True)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

Check if certain V features correlate strongly with fraud.

In [ ]:
corr_with_class = df.corr()['Class'].sort_values()

plt.figure(figsize=(10, 8))
corr_with_class.drop('Class').plot.barh(color='steelblue')
plt.title('Feature Correlation with Class (Fraud)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

print('Top 5 positively correlated with fraud:')
print(corr_with_class.sort_values(ascending=False).head(6))
print('\nTop 5 negatively correlated with fraud:')
print(corr_with_class.sort_values().head(5))

## Key Takeaways

1. **Severe class imbalance** — 0.17% fraud. SMOTE or class weighting is essential.
2. **No strong individual predictor** — max correlation with Class is ~0.3 (V14, V4). Need ensemble approach.
3. **Fraud amounts are small** — median fraud amount is much lower than legitimate. Card testing pattern.
4. **PCA features** — V1–V28 are already normalized. Only Time and Amount need scaling.
5. **Time patterns** — Fraud distribution across time differs slightly from legit. Worth keeping the feature.

Next: Train baseline models with SMOTE and compare performance.